In [1]:
import pandas as pd
import re

def parse_can_log(file_path):
    """Parse CAN log file and return a pandas DataFrame"""
    data = []
    
    with open(file_path, 'r') as f:
        for line in f:
            # Extract fields using regex
            match = re.search(r'Timestamp:\s([\d.]+)\s+ID:\s(\w+)\s+(\w+)\s+DLC:\s(\d+)\s+(.*)', line)
            if match:
                timestamp, can_id, flags, dlc, hex_data = match.groups()
                data.append({
                    'Timestamp': float(timestamp),
                    'ID': can_id,
                    'Flags': flags,
                    'DLC': int(dlc),
                    'Data': hex_data.strip()
                })
    
    df = pd.DataFrame(data)
    return df

# Usage
df = parse_can_log('dataset\\normal_run_data\\normal_run_data.txt')
print(df)

           Timestamp    ID Flags  DLC                     Data
0       1.479121e+09  0350   000    8  05 28 84 66 6d 00 00 a2
1       1.479121e+09  02c0   000    8  14 00 00 00 00 00 00 00
2       1.479121e+09  0430   000    8  00 00 00 00 00 00 00 00
3       1.479121e+09  04b1   000    8  00 00 00 00 00 00 00 00
4       1.479121e+09  01f1   000    8  00 00 00 00 00 00 00 00
...              ...   ...   ...  ...                      ...
988866  1.479122e+09  02b0   000    5           ac 05 0c 07 7f
988867  1.479122e+09  0316   000    8  05 38 10 0c 38 28 01 7a
988868  1.479122e+09  018f   000    8  fe 31 00 00 00 4b 00 00
988869  1.479122e+09  0260   000    8  32 38 39 30 ff 93 59 1c
988870  1.479122e+09  02a0   000    8  20 00 75 1d 01 04 dd 00

[988871 rows x 5 columns]


In [2]:
def parse_can_csv(file_path):
    # try reading with a header first
    df_DoS = pd.read_csv(file_path, skipinitialspace=True)
    
    if 'ID dlc' in df_DoS.columns:
        df_DoS[['ID', 'DLC']] = df_DoS['ID dlc'].astype(str).str.split(r'\s+', n=1, expand=True)
        df_DoS = df_DoS.drop(columns=['ID dlc'])
    elif set(['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','attack/nonattack']).issubset(df_DoS.columns):
        df_DoS = df_DoS.rename(columns={'attack/nonattack': 'Attack'})
    else:
        names = ['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','Attack']
        df_DoS = pd.read_csv(file_path, header=None, names=names, skipinitialspace=True)

    data_cols = [c for c in df_DoS.columns if c.lower().startswith('data')]
    df_DoS['Data'] = df_DoS[data_cols].astype(str).apply(
        lambda row: ' '.join(x for x in row if x not in ['', 'nan', 'NaN']),
        axis=1
    ).str.strip()

    df_DoS['Timestamp'] = pd.to_numeric(df_DoS['Timestamp'], errors='coerce')
    df_DoS['DLC'] = pd.to_numeric(df_DoS['DLC'], errors='coerce').astype('Int64')

    return df_DoS[['Timestamp','ID','DLC','Data','Attack']]




In [3]:
def parse_can_csv(file_path):
    df_can = pd.read_csv(file_path, skipinitialspace=True)

    if 'ID dlc' in df_can.columns:
        df_can[['ID', 'DLC']] = df_can['ID dlc'].astype(str).str.split(r'\s+', n=1, expand=True)
        df_can = df_can.drop(columns=['ID dlc'])
    elif set(['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','attack/nonattack']).issubset(df_can.columns):
        df_can = df_can.rename(columns={'attack/nonattack': 'Attack'})
    else:
        names = ['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','Attack']
        df_can = pd.read_csv(file_path, header=None, names=names, skipinitialspace=True)

    unnamed = [c for c in df_can.columns if isinstance(c, str) and c.startswith('Unnamed')]
    if 'Attack' not in df_can.columns and unnamed:
        df_can = df_can.rename(columns={unnamed[-1]: 'Attack'})

    data_cols = sorted(
        [c for c in df_can.columns if isinstance(c, str) and c.lower().startswith('data')],
        key=lambda x: int(re.search(r'\d+', x).group()) if re.search(r'\d+', x) else 0
    )

    def build_data(row):
        try:
            dlc = int(row['DLC'])
        except Exception:
            dlc = 0

        values = []
        for col in data_cols[:min(dlc, len(data_cols))]:
            val = row.get(col)
            if pd.isna(val):
                continue
            s = str(val).strip()
            if s.lower() in ['', 'nan']:
                continue
            values.append(s)

        return ' '.join(values).strip()

    def infer_attack(row):
        attack = row.get('Attack')
        if isinstance(attack, str) and attack.strip():
            return attack

        try:
            dlc = int(row['DLC'])
        except Exception:
            dlc = 0

        for col in data_cols[min(dlc, len(data_cols)):]:
            val = row.get(col)
            if pd.isna(val):
                continue
            s = str(val).strip()
            if s.lower() in ['', 'nan']:
                continue
            if not re.fullmatch(r'[0-9A-Fa-f]{2}', s):
                return s

        return pd.NA

    if data_cols:
        df_can['Data'] = df_can.apply(build_data, axis=1)
        if 'Attack' not in df_can.columns:
            df_can['Attack'] = pd.NA
        df_can['Attack'] = df_can.apply(infer_attack, axis=1).combine_first(df_can['Attack'])
    else:
        df_can['Data'] = df_can.get('Data', '').astype(str).str.strip()
        if 'Attack' not in df_can.columns:
            df_can['Attack'] = pd.NA

    df_can['Timestamp'] = pd.to_numeric(df_can['Timestamp'], errors='coerce')
    df_can['DLC'] = pd.to_numeric(df_can['DLC'], errors='coerce').astype('Int64')

    return df_can[['Timestamp','ID','DLC','Data','Attack']]

In [4]:
df_DoS = parse_can_csv(r'dataset\DoS_dataset.csv')
print(df_DoS.head())

      Timestamp    ID  DLC                     Data Attack
0  1.478198e+09  0316    8  05 21 68 09 21 21 00 6f      R
1  1.478198e+09  018f    8  fe 5b 00 00 00 3c 00 00      R
2  1.478198e+09  0260    8  19 21 22 30 08 8e 6d 3a      R
3  1.478198e+09  02a0    8  64 00 9a 1d 97 02 bd 00      R
4  1.478198e+09  0329    8  40 bb 7f 14 11 20 00 14      R


In [5]:
df_Fuzzy = parse_can_csv(r'dataset\Fuzzy_dataset.csv')
print(df_Fuzzy.head())

      Timestamp    ID  DLC                     Data Attack
0  1.478196e+09  0545    8  d8 00 00 8a 00 00 00 00      R
1  1.478196e+09  02b0    5           ff 7f 00 05 49      R
2  1.478196e+09  0002    8  00 00 00 00 00 01 07 15      R
3  1.478196e+09  0153    8  00 21 10 ff 00 ff 00 00      R
4  1.478196e+09  0130    8  19 80 00 ff fe 7f 07 60      R


In [6]:
df_gear = parse_can_csv(r'dataset\gear_dataset.csv')
print(df_gear.head())

      Timestamp    ID  DLC                     Data Attack
0  1.478193e+09  0140    8  00 00 00 00 10 29 2a 24      R
1  1.478193e+09  02c0    8  15 00 00 00 00 00 00 00      R
2  1.478193e+09  0350    8  05 20 44 68 77 00 00 7e      R
3  1.478193e+09  0370    8  00 20 00 00 00 00 00 00      R
4  1.478193e+09  043f    8  10 40 60 ff 78 c4 08 00      R


In [7]:
df_RPM = parse_can_csv(r'dataset\RPM_dataset.csv')
print(df_RPM.head())

      Timestamp    ID  DLC                     Data Attack
0  1.478191e+09  0316    8  05 22 68 09 22 20 00 75      R
1  1.478191e+09  018f    8  fe 3b 00 00 00 3c 00 00      R
2  1.478191e+09  0260    8  19 22 22 30 ff 8f 6e 3f      R
3  1.478191e+09  02a0    8  60 00 83 1d 96 02 bd 00      R
4  1.478191e+09  0329    8  dc b8 7e 14 11 20 00 14      R


In [8]:

# Normalize the normal dataset and add an Attack label of 'R'
df_normal = df[['Timestamp', 'ID', 'DLC', 'Data']].copy()
df_normal['Attack'] = 'R'

# Combine all datasets into one dataframe
df_combined = pd.concat(
    [
        df_normal,
        df_DoS[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']],
        df_Fuzzy[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']],
        df_gear[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']],
        df_RPM[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']],
    ],
    ignore_index=True
)

# Optional: keep columns in a consistent order
df_combined = df_combined[['Timestamp', 'ID', 'DLC', 'Data', 'Attack']]

print(df_combined.head())
print(df_combined['Attack'].value_counts())

      Timestamp    ID  DLC                     Data Attack
0  1.479121e+09  0350    8  05 28 84 66 6d 00 00 a2      R
1  1.479121e+09  02c0    8  14 00 00 00 00 00 00 00      R
2  1.479121e+09  0430    8  00 00 00 00 00 00 00 00      R
3  1.479121e+09  04b1    8  00 00 00 00 00 00 00 00      R
4  1.479121e+09  01f1    8  00 00 00 00 00 00 00 00      R
Attack
R    15226829
T     2331517
Name: count, dtype: int64


In [9]:
del df, df_normal, df_DoS, df_Fuzzy, df_gear, df_RPM



In [10]:
# sort by time and add neighbor ID columns and interval features
df_combined = df_combined.sort_values("Timestamp").reset_index(drop=True)

df_combined["Prev_ID"] = df_combined["ID"].shift(1)
df_combined["Next_ID"] = df_combined["ID"].shift(-1)
df_combined["Former_ID"] = df_combined["ID"].shift(2)   # before previous
df_combined["Latter_ID"] = df_combined["ID"].shift(-2)  # after next

df_combined["Prev_interval"] = df_combined["Timestamp"] - df_combined["Timestamp"].shift(1)
df_combined["Next_interval"] = df_combined["Timestamp"].shift(-1) - df_combined["Timestamp"]

# previous/next same-ID intervals
df_combined["_Prev_same_ts"] = df_combined.groupby("ID")["Timestamp"].shift(1)
df_combined["_Next_same_ts"] = df_combined.groupby("ID")["Timestamp"].shift(-1)

df_combined["Prev_same_ID_interval"] = df_combined["Timestamp"] - df_combined["_Prev_same_ts"]
df_combined["Next_same_ID_interval"] = df_combined["_Next_same_ts"] - df_combined["Timestamp"]

df_combined.drop(columns=["_Prev_same_ts", "_Next_same_ts"], inplace=True)

# show the new columns
print(df_combined[[
    "ID","Prev_ID","Next_ID","Former_ID","Latter_ID",
    "DLC","Prev_interval","Next_interval",
    "Prev_same_ID_interval","Next_same_ID_interval"
]].head())

     ID Prev_ID Next_ID Former_ID Latter_ID  DLC  Prev_interval  \
0  0316     NaN    018f       NaN      0260    8            NaN   
1  018f    0316    0260       NaN      02a0    8       0.000239   
2  0260    018f    02a0      0316      0329    8       0.000227   
3  02a0    0260    0329      018f      0545    8       0.000235   
4  0329    02a0    0545      0260      02c0    8       0.000228   

   Next_interval  Prev_same_ID_interval  Next_same_ID_interval  
0       0.000239                    NaN               0.010060  
1       0.000227                    NaN               0.010061  
2       0.000235                    NaN               0.010067  
3       0.000228                    NaN               0.010071  
4       0.000245                    NaN               0.010075  


In [15]:
import pandas as pd

# Only keep the columns you need
X = df_combined.loc[:8779174, [
    "ID","Prev_ID","Next_ID","Former_ID","Latter_ID",
    "DLC","Prev_interval","Next_interval",
    "Prev_same_ID_interval","Next_same_ID_interval"
]].copy()


def parse_hex_id(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return pd.NA
        if value.lower().startswith("0x"):
            value = value[2:]
        return int(value, 16)
    return int(value)

# Convert hexadecimal CAN IDs directly to nullable integers
for col in ["ID", "Prev_ID", "Next_ID", "Former_ID", "Latter_ID"]:
    X[col] = X[col].map(parse_hex_id).astype("UInt32")

# DLC only needs values 0-8
X["DLC"] = X["DLC"].astype("UInt8")


# Labels
y = df_combined["Attack"][:8779175].astype("category")

print(X.head())

    ID  Prev_ID  Next_ID  Former_ID  Latter_ID  DLC  Prev_interval  \
0  790     <NA>      399       <NA>        608    8            NaN   
1  399      790      608       <NA>        672    8       0.000239   
2  608      399      672        790        809    8       0.000227   
3  672      608      809        399       1349    8       0.000235   
4  809      672     1349        608        704    8       0.000228   

   Next_interval  Prev_same_ID_interval  Next_same_ID_interval  
0       0.000239                    NaN               0.010060  
1       0.000227                    NaN               0.010061  
2       0.000235                    NaN               0.010067  
3       0.000228                    NaN               0.010071  
4       0.000245                    NaN               0.010075  


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=10,
    random_state=42
)

rf.fit(X_train, y_train)

# Predict
predictions = rf.predict(X_test)

In [17]:
predictions
y_test

7679477    R
1040929    R
518981     T
7244829    R
1290334    R
          ..
557338     R
3190796    R
5260355    T
650488     R
1043203    R
Name: Attack, Length: 1755835, dtype: category
Categories (2, str): ['R', 'T']

In [18]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9983


In [ ]:
from sklearn.tree import export_text

print("Random Forest summary:")
print("n_estimators:", rf.n_estimators)
print("max_depth:", rf.max_depth)
print("min_samples_leaf:", rf.min_samples_leaf)
print("feature_importances:", dict(zip([
    "ID","Prev_ID","Next_ID","Former_ID","Latter_ID",
    "DLC","Prev_interval","Next_interval",
    "Prev_same_ID_interval","Next_same_ID_interval"
], rf.feature_importances_)))
print()

print("First tree structure (truncated):")
print(export_text(rf.estimators_[9], feature_names=[
    "ID","Prev_ID","Next_ID","Former_ID","Latter_ID",
    "DLC","Prev_interval","Next_interval",
    "Prev_same_ID_interval","Next_same_ID_interval"
], max_depth=10))

Random Forest summary:
n_estimators: 10
max_depth: None
min_samples_leaf: 1
feature_importances: {'ID': np.float64(0.05167846654637499), 'Prev_ID': np.float64(0.00693954243330493), 'Next_ID': np.float64(0.022919310366286333), 'Former_ID': np.float64(0.00734428454186801), 'Latter_ID': np.float64(0.0063079874038317486), 'DLC': np.float64(0.0022639262298167714), 'Prev_interval': np.float64(0.05164873890112065), 'Next_interval': np.float64(0.08472371864783217), 'Prev_same_ID_interval': np.float64(0.5514385147357547), 'Next_same_ID_interval': np.float64(0.21473551019380982)}

First tree structure (truncated):
|--- Prev_interval <= 0.00
|   |--- Next_interval <= 0.00
|   |   |--- Next_interval <= 0.00
|   |   |   |--- DLC <= 6.50
|   |   |   |   |--- class: 0.0
|   |   |   |--- DLC >  6.50
|   |   |   |   |--- Next_interval <= 0.00
|   |   |   |   |   |--- Former_ID <= 640.00
|   |   |   |   |   |   |--- Former_ID <= 503.50
|   |   |   |   |   |   |   |--- Prev_interval <= 0.00
|   |   |   |

: 